# Causal Retraining: A Deployable Model
### Retraining both models on features a live device can actually compute

**Why this notebook exists.** The shipped models in `Notebook_Improvements.ipynb` were trained on
features that cannot be computed in real time. Auditing the training code against the deployment
requirement revealed the conflict:

| Input | Source in training | Causal? |
|---|---|---|
| XGB `[0:13]` HRV features | window-local | Yes |
| XGB `[13:18]` residual features | whole-session Cosinor fit | **No** |
| XGB `[18:20]` MESOR, amplitude | whole-session Cosinor fit | **No** |
| XGB `[20:25]` circadian encodings | timestamp only | Yes |
| CNN ch.1 `rn` | session-wide mean/std | **No** |
| CNN ch.2 `rm` | centred window `rr[i-10:i+10]` | **No** (uses future) |
| CNN ch.3 `sd` | centred window | **No** (uses future) |
| CNN ch.4 `hr` | pointwise | Yes |
| CNN ch.5 `rrn` | Cosinor residual, session-normalised | **No** |
| CNN ch.6 `tn` | session-wide mean/std | **No** |
| CNN ch.7 `trn` | Cosinor residual, session-normalised | **No** |

Seven of 25 XGBoost features and six of seven CNN channels depend on information a streaming device
does not have. `docs/ARCHITECTURE.md` states the deployed system must be causal; the trained model
requires non-causal inputs. Both cannot hold.

**What this notebook does not do.** It does not patch the existing model to accept substituted
inputs. Feeding EWMA-derived features into a model trained on Cosinor-derived features would produce
a model whose reported F1 (0.682) describes a different feature set than the one it receives — an
unvalidated model wearing a validated model's number.

**What it does instead.** Every non-causal input is replaced with a causal equivalent, both models
are retrained from scratch on the new feature set, and the same leave-one-subject-out protocol is
re-run to produce an honest, deployable figure.

**Expected outcome.** Lower than 0.682. The scope-expansion work already measured the cost of causal
baseline estimation at roughly 0.06 macro-F1, recoverable to about 0.01 with multi-timescale
tracking. A comparable drop here would be consistent with that finding rather than a failure of this
notebook.


## 1. Setup

In [1]:
!pip install neurokit2 xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 688.9/688.9 kB 11.3 MB/s eta 0:00:00


In [2]:
import os, pickle, warnings, json, time
import numpy as np, pandas as pd
from scipy.signal import welch
try:
    from scipy.integrate import trapezoid as TRAPZ
except ImportError:
    from scipy.integrate import trapz as TRAPZ
from sklearn.model_selection import LeaveOneGroupOut
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import f1_score, cohen_kappa_score
from sklearn.utils.class_weight import compute_sample_weight, compute_class_weight
from xgboost import XGBClassifier
import tensorflow as tf
from tensorflow.keras import layers, callbacks, Model
from tensorflow.keras.losses import Loss
import neurokit2 as nk
warnings.filterwarnings('ignore'); np.random.seed(42); tf.random.set_seed(42)

DATA_PATH='/kaggle/input/datasets/orvile/wesad-wearable-stress-affect-detection-dataset/WESAD'
SAVE='/kaggle/working'; os.makedirs(SAVE,exist_ok=True)
SUBJECT_IDS=[2,3,4,5,6,7,8,9,10,11,13,14,15,16,17]
CLASS_NAMES=['relaxed','mild','moderate','high']; NCLS=4
WINDOW=120; STEP=5

# causal baseline configuration, matching docs/ARCHITECTURE.md
EWMA_HALFLIVES={'fast':60,'medium':300,'slow':1800}
POPULATION_RR_MS=780.0        # cold-start seed
ROLL_WINDOW=20                # causal rolling window, past-only

RESULTS=f'{SAVE}/causal_retrain.json'
print("TF",tf.__version__,"GPU",len(tf.config.list_physical_devices('GPU'))>0)

TF 2.20.0 GPU True


## 2. Data loading — unchanged from the original pipeline

RR extraction, cleaning and labelling are already causal (window-local or pointwise) and are copied
verbatim from `Notebook_Improvements.ipynb` cell 6.

In [3]:
def load_subject(sid):
    with open(f"{DATA_PATH}/S{sid}/S{sid}.pkl",'rb') as f:
        data=pickle.load(f,encoding='latin1')
    return data['signal']['chest'], data['signal']['wrist']['TEMP'].flatten(), data['label'].flatten()

def extract_rr_from_ecg(ecg, fs=700):
    ecg=nk.ecg_clean(ecg.flatten(), sampling_rate=fs)
    _,info=nk.ecg_peaks(ecg, sampling_rate=fs); rp=info['ECG_R_Peaks']
    return np.diff(rp)*(1000.0/fs), (rp[:-1]+rp[1:])/2.0/fs, rp

def clean_rr(rr,ts):
    rr=rr.copy().astype(float); rr[(rr<=300)|(rr>=2000)]=np.nan
    for i in range(1,len(rr)):
        if not np.isnan(rr[i-1]) and not np.isnan(rr[i]):
            if abs(rr[i]-rr[i-1])/rr[i-1]>0.20: rr[i]=np.nan
    m=np.isnan(rr)
    if m.any(): rr[m]=np.interp(np.where(m)[0],np.where(~m)[0],rr[~m])
    return rr, ts.copy()

def align_temp(wt, rp, fe=700, ft=4.0):
    tap=np.interp(rp/fe, np.arange(len(wt))/ft, wt); return (tap[:-1]+tap[1:])/2.0

def labels_to_rr(labels, rp):
    out=[]
    for i in range(len(rp)-1):
        seg=labels[rp[i]:rp[i+1]]; v=seg[seg>0]
        out.append(0 if len(v)==0 else np.bincount(v).argmax())
    return np.array(out)

wesad={}
for sid in SUBJECT_IDS:
    try:
        chest,wt,labels=load_subject(sid); ecg=chest['ECG'].flatten()
        rr,ts,rp=extract_rr_from_ecg(ecg); temp=align_temp(wt,rp); rr,ts=clean_rr(rr,ts)
        rl=labels_to_rr(labels,rp); keep=rl>0
        rrk,tk,tsk,lk=rr[keep],temp[keep],ts[keep],rl[keep]
        new=np.zeros(len(lk),dtype=int); si=np.where(lk==2)[0]
        if len(si)>0:
            srr=rrk[si]; loc=[]
            for i in range(len(srr)):
                w=srr[max(0,i-15):i+15]; dd=np.diff(w)
                loc.append(np.sqrt(np.mean(dd**2)) if len(dd)>0 else 50)
            loc=np.array(loc); p33,p66=np.percentile(loc,33),np.percentile(loc,66)
            for i,idx in enumerate(si): new[idx]=(1 if loc[i]>=p66 else 2 if loc[i]>=p33 else 3)
        wesad[f'S{sid}']=dict(rr_ms=rrk,temp=tk,timestamps=tsk,labels=new)
    except Exception as e: print("FAIL",sid,e)
print(len(wesad),"subjects"); assert len(wesad)==15

15 subjects


## 3. Causal replacements

Each function below replaces a non-causal counterpart. Every one is verified causal by construction
and by test in Section 4.

In [4]:
def ewma_causal(x, halflife):
    """Past-only exponentially weighted mean. Seeded from the population
    value so a new user has a defined baseline from beat one."""
    a=1-np.exp(np.log(0.5)/max(halflife,1))
    o=np.empty(len(x),dtype=float)
    state=float(POPULATION_RR_MS) if len(x)==0 else float(POPULATION_RR_MS)
    for i in range(len(x)):
        state=a*x[i]+(1-a)*state
        o[i]=state
    return o

def causal_zscore(x, halflife=300):
    """Replaces (x - session_mean)/session_std with a past-only equivalent.
    Running mean via EWMA; running SD via EWMA of squared deviation."""
    a=1-np.exp(np.log(0.5)/max(halflife,1))
    mu=np.empty(len(x)); sd=np.empty(len(x))
    m=float(x[0]) if len(x) else 0.0
    v=1.0
    for i in range(len(x)):
        d=x[i]-m
        m=m+a*d
        v=(1-a)*(v+a*d*d)
        mu[i]=m; sd[i]=np.sqrt(max(v,1e-8))
    return (x-mu)/(sd+1e-8)

def roll_rmssd_causal(x, w=ROLL_WINDOW):
    """Replaces the centred rr[i-10:i+10] window with a past-only window."""
    o=np.zeros(len(x))
    for i in range(len(x)):
        seg=x[max(0,i-w+1):i+1]
        o[i]=np.sqrt(np.mean(np.diff(seg)**2)) if len(seg)>1 else 0.0
    return o

def roll_sdnn_causal(x, w=ROLL_WINDOW):
    o=np.zeros(len(x))
    for i in range(len(x)):
        seg=x[max(0,i-w+1):i+1]
        o[i]=np.std(seg) if len(seg)>1 else 0.0
    return o

def multiscale_residuals(rr):
    """Replaces the whole-session Cosinor residual with deviation from
    three concurrent causal EWMA baselines (the configuration measured
    in the scope-expansion work to recover ~80% of the causal gap)."""
    return {k: rr-ewma_causal(rr,hl) for k,hl in EWMA_HALFLIVES.items()}
print("causal functions defined")

causal functions defined


### Feature builders

`hrv_features` and `circ_features` are transcribed **verbatim** from `Notebook_Improvements.ipynb`
cell 6, including the Welch grid starting at 0 and the unit-spacing `TRAPZ` call. These details
affect VLF/LF/HF magnitudes and must not be "corrected" — reproducing the original exactly is the
point.

In [5]:
def hrv_features(w, fs=4.0):
    """VERBATIM from Notebook_Improvements.ipynb cell 6. Do not modify.
    Note: Welch grid starts at 0 (not t[0]); TRAPZ takes no frequency
    argument (unit spacing). Both are intentional reproductions."""
    rr,diff=np.array(w),np.diff(w)
    mean_rr=np.mean(rr); sdnn=np.std(rr); rmssd=np.sqrt(np.mean(diff**2))
    pnn50=np.sum(np.abs(diff)>50)/len(diff)*100; cv=sdnn/mean_rr
    t=np.cumsum(rr)/1000.0; u=np.interp(np.arange(0,t[-1],1/fs),t,rr)
    fr,psd=welch(u,fs=fs,nperseg=min(256,len(u)))
    vlf=TRAPZ(psd[(fr>=0.003)&(fr<0.04)]); lf=TRAPZ(psd[(fr>=0.04)&(fr<0.15)])
    hf=TRAPZ(psd[(fr>=0.15)&(fr<0.40)])
    lf_hf=lf/(hf+1e-8); lf_nu=lf/(lf+hf+1e-8)
    sd1=np.sqrt(0.5)*np.std(diff); sd2=np.sqrt(max(2*sdnn**2-0.5*np.var(diff),0))
    sdr=sd1/(sd2+1e-8)
    return np.array([mean_rr,sdnn,rmssd,pnn50,cv,vlf,lf,hf,lf_hf,lf_nu,sd1,sd2,sdr])

def resid_features(rw):
    """VERBATIM from cell 6: mean, std, max|.|, polyfit slope, sum(r^2)/len.
    Note element order — an earlier port of this function had elements 3
    and 4 swapped and element 4 replaced with mean(abs(diff))."""
    r=np.array(rw)
    return np.array([np.mean(r),np.std(r),np.max(np.abs(r)),
                     np.polyfit(np.arange(len(r)),r,1)[0],
                     np.sum(r**2)/len(r)])

def circ_features(ts):
    """VERBATIM from cell 6. Pure function of timestamp, already causal."""
    t,hour=ts%86400,(ts%86400)/3600.0
    cort=0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2)
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
                     np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),cort])

def circ7(ts):
    """The CNN's 7-dim circadian vector (cell 10) — 5 base features plus
    two shifted harmonics. Also already causal."""
    t=ts%86400; hour=t/3600.0
    return np.array([np.sin(2*np.pi*t/86400),np.cos(2*np.pi*t/86400),
        np.sin(2*np.pi*t/5400),np.cos(2*np.pi*t/5400),
        0.6*np.exp(-0.5*((hour-8)/1.5)**2)+0.3*np.exp(-0.5*((hour-15)/1.5)**2),
        np.sin(2*np.pi*(hour-23)/24),np.cos(2*np.pi*(hour-23)/24)])
print("feature builders defined (hrv/resid/circ transcribed verbatim)")

feature builders defined (hrv/resid/circ transcribed verbatim)


## 4. Verify causality before building anything

Each causal function is tested by corrupting the second half of a real subject's signal and
confirming the first half of its output is unchanged. A function that fails this is not causal
regardless of what its name suggests.

In [6]:
sig=wesad['S2']['rr_ms'].copy()
half=len(sig)//2
corrupt=sig.copy(); corrupt[half:]=9999.0

tests={
 'ewma_causal(fast)'  : lambda x: ewma_causal(x,60),
 'ewma_causal(medium)': lambda x: ewma_causal(x,300),
 'ewma_causal(slow)'  : lambda x: ewma_causal(x,1800),
 'causal_zscore'      : lambda x: causal_zscore(x),
 'roll_rmssd_causal'  : lambda x: roll_rmssd_causal(x),
 'roll_sdnn_causal'   : lambda x: roll_sdnn_causal(x),
}
print("="*66)
print("CAUSALITY VERIFICATION")
print("="*66)
print(f"{'function':<24}{'first half identical':<24}{'verdict'}")
print("-"*66)
allok=True
for name,fn in tests.items():
    a=fn(sig); b=fn(corrupt)
    ok=np.allclose(a[:half],b[:half])
    allok&=ok
    print(f"{name:<24}{str(ok):<24}{'OK' if ok else 'FAIL'}")
print("-"*66)
print("ALL CAUSAL" if allok else "FAILURE - do not proceed")
assert allok, "a function claimed causal is not causal"

CAUSALITY VERIFICATION
function                first half identical    verdict
------------------------------------------------------------------
ewma_causal(fast)       True                    OK
ewma_causal(medium)     True                    OK
ewma_causal(slow)       True                    OK
causal_zscore           True                    OK
roll_rmssd_causal       True                    OK
roll_sdnn_causal        True                    OK
------------------------------------------------------------------
ALL CAUSAL


## 5. Build the causal feature set

Structure matches `build_all` in cell 10 exactly. Only the *sources* of the non-causal inputs
change.

**XGBoost 25-vector.** The 5 residual features now derive from the medium-scale causal EWMA
residual rather than the Cosinor residual. The 2 Cosinor parameters (MESOR, amplitude) have no
causal equivalent — a whole-session rhythm fit cannot be computed from a buffer — and are replaced
with the current fast and slow EWMA baseline levels, which are the closest causal analogue: a
short-term and a long-term estimate of the person's expected level.

**CNN 7 channels.** Session-wide normalisation becomes causal z-scoring; centred rolling windows
become past-only windows; Cosinor residuals become EWMA residuals.

In [7]:
XGB_NAMES=(['mean_RR','SDNN','RMSSD','pNN50','CV_RR','VLF','LF','HF','LF/HF','LF_nu',
            'SD1','SD2','SD1/SD2']
         + ['res_mean','res_SD','res_maxabs','res_slope','res_msq']
         + ['ewma_fast_level','ewma_slow_level']
         + ['sin_24h','cos_24h','sin_90m','cos_90m','cortisol'])
assert len(XGB_NAMES)==25

def build_causal(window=WINDOW, step=STEP):
    Xseq,Xcirc,Xxgb,y,g=[],[],[],[],[]
    for sid,d in wesad.items():
        rr,temp,labels,ts=d['rr_ms'],d['temp'],d['labels'],d['timestamps']

        # causal baselines and residuals
        base=  {k:ewma_causal(rr,hl) for k,hl in EWMA_HALFLIVES.items()}
        res_med = rr-base['medium']

        # causal temperature baseline (same treatment)
        tbase_med = ewma_causal(temp, EWMA_HALFLIVES['medium'])
        temp_res  = temp-tbase_med

        # CNN channels, all causal
        rn  = causal_zscore(rr)
        rm  = roll_rmssd_causal(rn)
        sd  = roll_sdnn_causal(rn)
        hr  = 60000/(rr+1e-8)
        rrn = causal_zscore(res_med)
        tn  = causal_zscore(temp)
        trn = causal_zscore(temp_res)

        for s in range(0,len(rr)-window,step):
            e=s+window; mid=s+window//2; bi=min(mid,len(ts)-1)
            seq=np.stack([rn[s:e],rm[s:e],sd[s:e],hr[s:e],rrn[s:e],tn[s:e],trn[s:e]],axis=-1)
            try:
                xgbf=np.concatenate([
                    hrv_features(rr[s:e]),
                    resid_features(res_med[s:e]),
                    np.array([base['fast'][e-1], base['slow'][e-1]]),   # causal level estimates
                    circ_features(ts[bi])])
            except Exception:
                continue
            Xseq.append(seq); Xcirc.append(circ7(ts[bi])); Xxgb.append(xgbf)
            y.append(labels[mid]); g.append(int(sid[1:]))
    return (np.array(Xseq,np.float32),np.array(Xcirc,np.float32),
            np.array(Xxgb),np.array(y,np.int32),np.array(g,np.int32))

t0=time.time()
X_seq,X_circ,X_xgb,y_all,groups=build_causal()
print(f"built in {(time.time()-t0)/60:.1f} min")
print("seq",X_seq.shape,"circ",X_circ.shape,"xgb",X_xgb.shape)
print("classes",np.bincount(y_all))
assert X_xgb.shape[1]==25, f"expected 25 XGB features, got {X_xgb.shape[1]}"
assert X_seq.shape[1:]==(120,7) and X_circ.shape[1]==7
print("\nshapes match the original training contract")

built in 0.2 min
seq (11846, 120, 7) circ (11846, 7) xgb (11846, 25)
classes [8594 1101 1080 1071]

shapes match the original training contract


## 6. Model definitions — verbatim from cell 12

Architecture is unchanged. Only the data feeding it differs.

In [8]:
class SparseFocalLoss(Loss):
    def __init__(self, gamma=2.0):
        super().__init__(); self.gamma=gamma
    def call(self,yt,yp):
        yt=tf.cast(yt,tf.int32)
        ce=tf.keras.losses.sparse_categorical_crossentropy(yt,yp)
        pt=tf.reduce_sum(tf.one_hot(yt,4)*yp,axis=-1)
        return tf.pow(1.0-pt,self.gamma)*ce

def build_cnn(window=120,nch=7,ncirc=7,ncls=4):
    si=tf.keras.Input(shape=(window,nch),name='sequence')
    ci=tf.keras.Input(shape=(ncirc,),name='circadian')
    x=layers.Conv1D(64,7,padding='same',activation='relu')(si)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Conv1D(128,5,padding='same',activation='relu')(x)
    x=layers.BatchNormalization()(x); x=layers.MaxPooling1D(2)(x)
    x=layers.Bidirectional(layers.LSTM(128,return_sequences=True))(x)
    x=layers.Dropout(0.4)(x); a=layers.Attention()([x,x]); x=layers.GlobalAveragePooling1D()(a)
    c=layers.Dense(32,activation='relu')(ci); c=layers.Dense(16,activation='relu')(c)
    x=layers.Concatenate()([x,c]); x=layers.Dense(64,activation='relu')(x); x=layers.Dropout(0.4)(x)
    out=layers.Dense(ncls,activation='softmax')(x)
    return Model([si,ci],out,name='CNN_BiLSTM_Attn_causal')

def make_xgb():
    return XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.05,subsample=0.8,
        colsample_bytree=0.8,objective='multi:softprob',num_class=4,
        eval_metric='mlogloss',random_state=42,n_jobs=-1)
print("models defined")

models defined


## 7. Leave-one-subject-out retraining

Identical protocol to cell 14, including the 0.45/0.55 blend. The only change is the feature set.

In [9]:
logo=LeaveOneGroupOut(); store=[]
print("Retraining on causal features...\n")
t0=time.time()
for tr,te in logo.split(X_seq,y_all,groups):
    s=int(np.unique(groups[te])[0]); print(f"S{s:02d}",end=' ',flush=True)
    sc=StandardScaler(); Xtr=sc.fit_transform(X_xgb[tr]); Xte=sc.transform(X_xgb[te])
    sw=compute_sample_weight('balanced',y_all[tr])
    xgb=make_xgb(); xgb.fit(Xtr,y_all[tr],sample_weight=sw,verbose=False)
    p_xgb=xgb.predict_proba(Xte)

    cw=compute_class_weight('balanced',classes=np.unique(y_all[tr]),y=y_all[tr])
    cwd=dict(enumerate(cw))
    cnn=build_cnn(); cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-4),
                                 loss=SparseFocalLoss(2.0),metrics=['accuracy'])
    cb=[callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True,verbose=0),
        callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,verbose=0)]
    cnn.fit([X_seq[tr],X_circ[tr]],y_all[tr],validation_split=0.15,epochs=120,batch_size=32,
            class_weight=cwd,callbacks=cb,verbose=0)
    p_cnn=cnn.predict([X_seq[te],X_circ[te]],verbose=0)

    store.append({'sub':s,'y':y_all[te].tolist(),
                  'p_xgb':p_xgb.tolist(),'p_cnn':p_cnn.tolist()})
    f=f1_score(y_all[te],np.argmax(0.45*p_xgb+0.55*p_cnn,axis=1),average='macro',zero_division=0)
    print(f"f1={f:.3f}")
    tf.keras.backend.clear_session()

json.dump(store,open(RESULTS,'w'))
print(f"\nelapsed {(time.time()-t0)/60:.1f} min  ->  saved {RESULTS}")

Retraining on causal features...

S02 

I0000 00:00:1786443399.480507      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 13756 MB memory:  -> device: 0, name: Tesla T4, pci bus id: 0000:00:04.0, compute capability: 7.5
I0000 00:00:1786443399.483220      22 gpu_device.cc:2020] Created device /job:localhost/replica:0/task:0/device:GPU:1 with 13756 MB memory:  -> device: 1, name: Tesla T4, pci bus id: 0000:00:05.0, compute capability: 7.5


f1=0.452
S03 f1=0.652
S04 f1=0.373
S05 f1=0.346
S06 f1=0.597
S07 f1=0.595
S08 f1=0.558
S09 f1=0.275
S10 f1=0.691
S11 f1=0.662
S13 f1=0.663
S14 f1=0.643
S15 f1=0.216
S16 f1=0.699
S17 f1=0.588

elapsed 22.8 min  ->  saved /kaggle/working/causal_retrain.json


In [10]:
store=json.load(open(RESULTS))
for f in store:
    f['y']=np.array(f['y']); f['p_xgb']=np.array(f['p_xgb']); f['p_cnn']=np.array(f['p_cnn'])

def agg(w_xgb,w_cnn):
    yt,yp=[],[]
    for fs in store:
        yp.extend(np.argmax(w_xgb*fs['p_xgb']+w_cnn*fs['p_cnn'],axis=1)); yt.extend(fs['y'])
    yt,yp=np.array(yt),np.array(yp)
    return (float(np.mean(yt==yp)),
            f1_score(yt,yp,average='macro',zero_division=0),
            cohen_kappa_score(yt,yp,weights='quadratic'))


# add after the agg() definition in Section 7
best=(None,-1)
for w in np.arange(0,1.01,0.05):
    _,f,_=agg(w,1-w)
    if f>best[1]: best=(w,f)
print(f"best causal blend: xgb={best[0]:.2f} cnn={1-best[0]:.2f} -> F1={best[1]:.4f}")
print(f"XGBoost alone: {agg(1.0,0.0)[1]:.4f}")

acc,f1,kap=agg(0.45,0.55)
xgb_only=agg(1.0,0.0); cnn_only=agg(0.0,1.0)

print("="*68)
print("CAUSAL MODEL — LOSO RESULTS")
print("="*68)
print(f"{'configuration':<34}{'Acc':>9}{'Macro F1':>11}{'kappa':>9}")
print("-"*68)
print(f"{'XGBoost alone (causal)':<34}{xgb_only[0]:>9.3f}{xgb_only[1]:>11.3f}{xgb_only[2]:>9.3f}")
print(f"{'CNN alone (causal)':<34}{cnn_only[0]:>9.3f}{cnn_only[1]:>11.3f}{cnn_only[2]:>9.3f}")
print(f"{'Two-way ensemble 0.45/0.55':<34}{acc:>9.3f}{f1:>11.3f}{kap:>9.3f}")
print("="*68)
print()
print("REFERENCE — non-causal model (Notebook_Improvements.ipynb cell 14):")
print(f"{'  two-way ensemble':<34}{0.872:>9.3f}{0.682:>11.3f}{0.855:>9.3f}")
print()
print(f"causality cost: {0.682-f1:+.3f} macro-F1")
print()
print("The scope-expansion work measured the cost of causal baseline")
print("estimation at roughly 0.06 F1 with a single tracker, recoverable to")
print("about 0.01 with multi-timescale tracking. A drop in that range is")
print("consistent with that finding. A much larger drop suggests the")
print("Cosinor parameters carried more than baseline level alone.")

best causal blend: xgb=0.50 cnn=0.50 -> F1=0.5975
XGBoost alone: 0.5811
CAUSAL MODEL — LOSO RESULTS
configuration                           Acc   Macro F1    kappa
--------------------------------------------------------------------
XGBoost alone (causal)                0.823      0.581    0.774
CNN alone (causal)                    0.804      0.537    0.721
Two-way ensemble 0.45/0.55            0.838      0.596    0.786

REFERENCE — non-causal model (Notebook_Improvements.ipynb cell 14):
  two-way ensemble                    0.872      0.682    0.855

causality cost: +0.086 macro-F1

The scope-expansion work measured the cost of causal baseline
estimation at roughly 0.06 F1 with a single tracker, recoverable to
about 0.01 with multi-timescale tracking. A drop in that range is
consistent with that finding. A much larger drop suggests the
Cosinor parameters carried more than baseline level alone.


## 8. Export deployable artifacts

Trained on **all 15 subjects** — this is the model that ships, not a fold model. The scaler must be
the exact object fitted here; normalising live data with different statistics degrades predictions
silently.

In [11]:
sc_final=StandardScaler().fit(X_xgb)
xgb_final=make_xgb()
xgb_final.fit(sc_final.transform(X_xgb),y_all,
              sample_weight=compute_sample_weight('balanced',y_all),verbose=False)

cw=compute_class_weight('balanced',classes=np.unique(y_all),y=y_all)
cnn_final=build_cnn()
cnn_final.compile(optimizer=tf.keras.optimizers.Adam(1e-4),loss=SparseFocalLoss(2.0),metrics=['accuracy'])
cb=[callbacks.EarlyStopping(monitor='val_loss',patience=15,restore_best_weights=True,verbose=0),
    callbacks.ReduceLROnPlateau(monitor='val_loss',factor=0.5,patience=7,verbose=0)]
cnn_final.fit([X_seq,X_circ],y_all,validation_split=0.15,epochs=120,batch_size=32,
              class_weight=dict(enumerate(cw)),callbacks=cb,verbose=0)

cnn_final.save(f'{SAVE}/cnn_population.keras')
xgb_final.save_model(f'{SAVE}/xgb_population.json')
with open(f'{SAVE}/feature_scaler.pkl','wb') as fh: pickle.dump(sc_final,fh)

cfg=dict(
  model_type='two-way causal ensemble',
  window_beats=WINDOW, step_beats=STEP,
  ensemble_weights={'xgb':0.45,'cnn':0.55},
  ewma_halflives=EWMA_HALFLIVES,
  population_rr_ms=POPULATION_RR_MS,
  roll_window=ROLL_WINDOW,
  xgb_feature_order=XGB_NAMES,
  cnn_sequence_channels=['rn','rm','sd','hr','rrn','tn','trn'],
  cnn_circadian_dim=7,
  class_names=CLASS_NAMES,
  loso_macro_f1=float(f1), loso_kappa=float(kap),
  note=('Trained on CAUSAL features only. Supersedes the Cosinor-based '
        'model, whose features cannot be computed in real time.'))
json.dump(cfg,open(f'{SAVE}/model_config.json','w'),indent=2)

print("exported:")
for f in ['cnn_population.keras','xgb_population.json','feature_scaler.pkl','model_config.json']:
    print(f"  {f}  ({os.path.getsize(f'{SAVE}/{f}')/1e6:.2f} MB)")
print("\nDownload these into artifacts/ — models/, scalers/, config/ respectively.")

exported:
  cnn_population.keras  (4.01 MB)
  xgb_population.json  (4.74 MB)
  feature_scaler.pkl  (0.00 MB)
  model_config.json  (0.00 MB)

Download these into artifacts/ — models/, scalers/, config/ respectively.


## 9. What to do with this result

**If the causality cost is around 0.06 or less**, this is the deployable model. Export the
artifacts, update `docs/ARCHITECTURE.md` with the new figure, and note that the previously reported
0.682 describes a non-causal configuration that was never deployable.

**If the cost is substantially larger**, the Cosinor parameters were carrying more than a baseline
level — most likely the whole-session normalisation itself was doing significant work. In that case
the honest options are to report the causal figure as the deployable result regardless, or to
reconsider whether the system needs a warm-up period before it begins predicting.

**What to report in the paper.** Nothing here changes the mechanism findings. It changes which model
is described as deployable. The ICATC paper's framing — that offline analysis uses information a
deployed device does not have, and this has a measurable cost — is directly supported by this
result and can cite it.

**One caveat to state.** The MESOR and amplitude replacements (fast and slow EWMA levels) are
analogues, not equivalents. A whole-session rhythm fit and a running level estimate are different
quantities. The retrained model is a different model, evaluated honestly, rather than the same model
made causal.
